# Computational Theory: SHA-256

## Problem 1: Representing SHA-256 Data

### 32-bit words
Before beginning with any proper SHA-256 implementation starting with looking into how the input, output and intermediate data can be represented in Python.

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §2.1: defines a word as a group of 32 bits for SHA-256.
- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §3.2: words are added modulo 2<sup>32</sup>, which is why the sum below is masked back to 32 bits.

In [1]:
# 32-bit words.

# largest 32-bit value.
mask = 0xFFFFFFFF

# The largest unsigned 32-bit value, and 1 to add to it.
hello = 0xFFFFFFFF
bye = 1

# The sum is 0x100000000, which needs 33 bits. ANDing it with the mask
# drops bit 32, so the result wraps to 0.
result = (hello + bye) & mask

# Printing value as hex.
print(f"0x{result:08X}")

0x00000000


### Sequences of 32-bit words

SHA-256 works with 32 bit words the whole way through. Everything it uses comes in fixed size groups of them: the 16 words of a message block, the 64-word message schedule, the 64 round constants, and the 8 words of the hash state.

I'll represent each of these as a Python list of ints. Every element is an unlimited size
integer, so each one has to be masked manually to stay within 32 bits.

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §5.2.1: each 512-bit message block is split into sixteen 32-bit words.
- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §4.2.2: the sixty-four 32-bit round constants.
- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §5.3.3: the initial hash value, made of eight 32-bit words.
- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §6.2.2: the 64-word message schedule and the hash computation.

In [2]:
# Example of eight random 32 bit words.
words = [
    0x1a2b3c4d, 0xdeadbeef, 0x00000001, 0xffffffff,
    0x80000000, 0x12345678, 0x26d8aabe, 0x0badf00d,
]

# Add each word to itself, masking so we keep 32 bits.
doubled = [(w + w) & mask for w in words]

# Print each value as hex
print([f"{w:08x}" for w in doubled])

['3456789a', 'bd5b7dde', '00000002', 'fffffffe', '00000000', '2468acf0', '4db1557c', '175be01a']


You can use a Python list of ints to represent sequences of 32 bit words, but since Python ints can grow large, you have to apply a mask after every operation to keep the values within 32 bits.

### Input Messages

SHA-256 takes a message as a sequence of bytes, I'll use Python bytes objects to represent input messages. Text has to be encoded first. UTF-8 turns ASCII characters into one byte each and other characters into several bytes.

**References**

- [Python docs: `str.encode`](https://docs.python.org/3/library/stdtypes.html#str.encode): converts a string to bytes using a given encoding.
- [RFC 3629: UTF-8](https://www.rfc-editor.org/rfc/rfc3629): the UTF-8 standard, which encodes ASCII characters as one byte and other characters as two to four bytes.

In [3]:
# Encode text messages as UTF-8 bytes.
ascii_msg = "hello".encode("utf-8")
# "é" is not an ASCII character, so UTF-8 encodes it as two bytes.
non_ascii_msg = "café".encode("utf-8")

# Show each message's bytes in hex, then compare how many characters it has
# with how many bytes (and bits) it takes up.
for msg in [ascii_msg, non_ascii_msg]:
    chars = len(msg.decode("utf-8"))
    print(msg, msg.hex(), f"{chars} characters, {len(msg)} bytes, {len(msg) * 8} bits")

b'hello' 68656c6c6f 5 characters, 5 bytes, 40 bits
b'caf\xc3\xa9' 636166c3a9 4 characters, 5 bytes, 40 bits


bytes is the right fit because SHA-256 hashes bytes and not characters. It's
immutable, its length in bits is just `len(msg) * 8`, and it's what `hashlib.sha256` accepts, so I can test against it.

**References**

- [Python docs: `bytes`](https://docs.python.org/3/library/stdtypes.html#bytes): Python's immutable sequence of bytes.
- [Python docs: `hashlib`](https://docs.python.org/3/library/hashlib.html): Python's built-in SHA-256, used to check results.

### 512-bit message blocks

SHA-256 processes a message in 512 bit blocks. That's 64 bytes, or 16
32-bit words. I can represent a block as a list of 16 ints, built by splitting
64 bytes into 4 byte chunks and converting each chunk to a word.

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §5.2.1: the padded message is parsed into 512-bit blocks of sixteen 32-bit words.

In [4]:
# A 64 byte block
block = bytes(range(64))

# Split into 4 byte chunks and then we convert each one to a 32 bit word.
block_words = [int.from_bytes(block[i:i + 4], "big") for i in range(0, 64, 4)]

print(f"{len(block) * 8} bits, {len(block_words)} words")
print([f"{w:08x}" for w in block_words])

512 bits, 16 words
['00010203', '04050607', '08090a0b', '0c0d0e0f', '10111213', '14151617', '18191a1b', '1c1d1e1f', '20212223', '24252627', '28292a2b', '2c2d2e2f', '30313233', '34353637', '38393a3b', '3c3d3e3f']


A list of 16 words matches how the standard describes a block. Real messages are almost never a multiple of 64 bytes.

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §5.1.1: how a message is padded to a multiple of 512 bits before it is split into blocks.

### The final 256-bit hash value

SHA-256 ends with eight 32 bit words of hash state. Put
together, they make up the 256-bit digest. I'll keep it as a list of 8 words while
computing, then can return it as a 64 character lowercase hex string.

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §6.2.2: the final digest is the eight hash words H<sub>0</sub> to H<sub>7</sub> joined together.
- [NIST SHA-256 example values](https://csrc.nist.gov/CSRC/media/Projects/Cryptographic-Standards-and-Guidelines/documents/examples/SHA256.pdf): the worked example for `"abc"` that the expected digest below comes from.

In [5]:
# Final hash state for the message "abc".
final_words = [
    0xba7816bf, 0x8f01cfea, 0x414140de, 0x5dae2223,
    0xb00361a3, 0x96177a9c, 0xb410ff61, 0xf20015ad,
]

# Join the words as 8 digit hex strings so we can get the 64 character digest.
digest = "".join(f"{w:08x}" for w in final_words)

print(digest, f"({len(digest)} hex characters)")

# Expected SHA-256 digest of 'abc' from the standards examples.
assert digest == "ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad"

ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad (64 hex characters)


Eight words are handy while computing, since each one gets updated separately.
A hex string is also the standard way to show a digest since each hex digit is 4 bits, so
256 bits is always 64 characters.

### Big-endian integers

SHA-256 reads and writes words in big-endian order, the most significant byte
comes first. Python's `int.from_bytes` and
`int.to_bytes` take the byte order as an argument, so I can easily pass `"big"`.

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §3.1: the standard uses the big-endian convention for words.
- [Python docs: `int.from_bytes`](https://docs.python.org/3/library/stdtypes.html#int.from_bytes) and [`int.to_bytes`](https://docs.python.org/3/library/stdtypes.html#int.to_bytes): convert between bytes and ints with a chosen byte order.

In [6]:
# Four bytes in order: 0x01, 0x02, 0x03, 0x04.
chunk = bytes([0x01, 0x02, 0x03, 0x04])

# Read the same bytes both ways.
big = int.from_bytes(chunk, "big")
little = int.from_bytes(chunk, "little")

print(f"big-endian:    0x{big:08x}")
print(f"little-endian: 0x{little:08x}")

# Convert back with the same byte order and you get the original bytes.
assert big.to_bytes(4, "big") == chunk

big-endian:    0x01020304
little-endian: 0x04030201


Byte order matters since the same bytes can give you different words. Most
computers are little-endian, so the order has to be stated directly.
If you read a message with the wrong order then you'd get a hash that looks fine but is
completely different.

## Problem 2: SHA-256 Bitwise Operations

SHA-256 applies six bitwise functions to each message block. All of them operate on 32-bit words, so I'll use NumPy's uint32 type to keep values at that size.

The functions below follow the definitions in FIPS 180-4. ROTR stands for rotate right, meaning bits shifted off the right side wrap around to the left. A normal right shift (>>) doesn't wrap, so the rotation combines a right shift with a left shift.

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §4.1.2: defines the SHA-256 logical functions.
- [NumPy docs: `numpy.uint32`](https://numpy.org/doc/stable/reference/arrays.scalars.html#numpy.uint32): the unsigned 32-bit integer type.

### Choice function: Ch(x, y, z)

The choice function uses x to decide which bits to take from y or z. If a bit in x is 1 it takes the matching bit from y. If the bit is 0 it takes the bit from z.

Defined as:

$$\text{Ch}(x, y, z) = (x \land y) \oplus (\lnot x \land z)$$

Here $\land$ is bitwise AND (`&`), $\lnot$ is bitwise NOT (`~`) and $\oplus$ is bitwise XOR (`^`).

In [7]:
import numpy as np  # NumPy gives us the uint32 type for 32-bit words

def Ch(x, y, z):
    """SHA-256 choice function for three 32-bit words."""
    # Force all three inputs to 32 bits so the bitwise math matches SHA-256
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)

    # For each bit, take from y where x is 1 and from z where x is 0
    return (x & y) ^ (~x & z)


# First test. x is all 1s so every bit picks from y, and the result should be y.
x = np.uint32(0xFFFFFFFF)
y = np.uint32(0x12345678)
z = np.uint32(0xABCDEF01)

# Check the result matches y
assert Ch(x, y, z) == y
# Print the result as 8 hex digits
print(f"Ch: 0x{Ch(x, y, z):08x}")

# Second test. x is all 0s so every bit picks from z, and the result should be z.
assert Ch(np.uint32(0), y, z) == z
# Print the result as 8 hex digits
print(f"Ch with x = 0: 0x{Ch(0, y, z):08x}")

Ch: 0x12345678
Ch with x = 0: 0xabcdef01


### Majority function: Maj(x, y, z)

The majority function looks at each bit position in the three inputs. If two or more of those bits are 1, the result bit is 1. If not, it's 0.

Defined as:

$$\text{Maj}(x, y, z) = (x \land y) \oplus (x \land z) \oplus (y \land z)$$

The function is symmetrical because it checks every possible pair of inputs. Changing the order of x, y and z does not change the result.

In SHA-256, Maj is used in all 64 compression rounds. It takes the three working variables a, b and c as inputs and helps compute T2.

In [8]:
import numpy as np  # NumPy gives us the uint32 type for 32-bit words

def Maj(x, y, z):
    """SHA-256 majority function for three 32-bit words."""
    # Force all three inputs to 32 bits so the bitwise math matches SHA-256
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)

    # A bit is 1 when it shows up in at least two of the three inputs
    return (x & y) ^ (x & z) ^ (y & z)


# First test. All three inputs are the same, so the result should match them.
x = np.uint32(0x12345678)
# Check the result matches the shared value
assert Maj(x, x, x) == x
# Print the result as 8 hex digits
print(f"Maj: 0x{Maj(x, x, x):08x}")

# Second test. Only one input has 1 bits, so no bit can have a majority
assert Maj(0xFFFFFFFF, 0, 0) == np.uint32(0)
# Print the result as 8 hex digits
print(f"Maj with two zero inputs: 0x{Maj(0xFFFFFFFF, 0, 0):08x}")

Maj: 0x12345678
Maj with two zero inputs: 0x00000000


### Uppercase sigma 0 function: Sigma0(x)

Sigma0 takes a 32-bit input and rotates its bits right by 2, 13 and 22 positions. It then XORs the three results to get a final value.

A normal right shift drops bits off the right side. Rotation does not. Bits that leave the right side come back on the left.

Defined as:

$$\Sigma_0^{\{256\}}(x) = \text{ROTR}^{2}(x) \oplus \text{ROTR}^{13}(x) \oplus \text{ROTR}^{22}(x)$$

The ROTR helper function below does the rotation using NumPy shifts and bitwise OR.

In [9]:
import numpy as np  # NumPy gives us the uint32 type for 32-bit words

def ROTR(x, n):
    """Rotate a 32-bit unsigned integer right by n bit positions."""
    # Force the input to 32 bits and the shift count to a plain int
    x = np.uint32(x)
    n = int(n)

    # Shift right to drop the low bits, shift left to bring them back on top
    return np.uint32((x >> np.uint32(n)) | (x << np.uint32(32 - n)))


def Sigma0(x):
    """SHA-256 uppercase sigma 0 function."""
    # Force the input to 32 bits
    x = np.uint32(x)

    # XOR the three rotated copies of x
    return ROTR(x, 2) ^ ROTR(x, 13) ^ ROTR(x, 22)


# Test the rotation helper with a value that has the top bit set.
rotated = ROTR(np.uint32(0x80000001), 1)
# Rotating right by 1 should wrap the low bit to the top and shift the rest
assert rotated == np.uint32(0xC0000000)
# Print the result as 8 hex digits
print(f"ROTR test: 0x{rotated:08x}")

# Test Sigma0 with a 32-bit input
x = np.uint32(0x12345678)
# Print the result as 8 hex digits
print(f"Sigma0: 0x{Sigma0(x):08x}")
# Check the result is still a 32-bit unsigned integer
assert isinstance(Sigma0(x), np.uint32)

ROTR test: 0xc0000000
Sigma0: 0x66146474


The function rotates the same input three times, each with a different number of positions.
The first rotation moves every bit 2 positions to the right. The second moves every bit 13 positions, and the third moves every bit 22 positions.
Using different rotation amounts means each output bit is influenced by different positions in the original input.
The three rotated values are then combined using XOR. XOR gives a 1 when an odd number of the matching input bits are 1, otherwise it gives 0.
Because rotations keep all 32 bits, no bits are lost during the individual rotations. The XOR then combines the different bit positions into one result.